<h1><center>Data Science RAG</center></h1>

**A Retrieval-Augmented Generation (RAG) assistant for data-science documentation.**

This notebook builds a question-answering system over the official reference docs (~1,040 PDF pages) of six libraries: **pandas, NumPy, SciPy, scikit-learn, Matplotlib, and Plotly**.

**How it works (RAG in one paragraph):** instead of asking an LLM to answer from memory (where it can hallucinate), we first *retrieve* the most relevant documentation passages for a question, then hand only those passages to the LLM and require it to answer *from them*, with citations. To make retrieval fast, every chunk of documentation is converted into an *embedding* (a 1,536-number vector that captures its meaning) and stored in a vector database; a question is embedded the same way and matched to chunks by cosine similarity.

**Stack:** PyMuPDF (PDF extraction) / LangChain (splitting, prompts) / OpenAI `text-embedding-3-large` @ 1,536 dims (embeddings) / Supabase Postgres + pgvector (vector store) / `gpt-6-luna` (answering) / Plotly (diagnostics).

### Build plan & status

| Step | What | Status |
|---|---|---|
| 1 | Config cell (models, constants, sanity checks) | done |
| 2 | Supabase schema (`documents` table, HNSW index, `match_documents`) | done (in Supabase SQL) |
| 3 | Load & clean PDFs with metadata | done |
| 4 | Chunking + chunk-quality analysis | done |
| 5 | Embed + upsert in batches | done |
| 6 | Retrieval sanity test | done |
| 7 | t-SNE embedding visualization | done |
| 8 | RAG chain with structured citations | done |
| 9 | Evaluation: hit rate@k and MRR | done |
| 10 | Streamlit app (chat, citations, library filter) | next |

> The detailed build log (decisions, bugs found, lessons learned) lives in [sessions.md](./sessions.md).


## Setup — Imports & Configuration (Steps 1–2)

**What this does:** imports everything the pipeline needs, sets the knobs for the whole notebook in one place (model names, embedding size, table name, chunk sizes), fails fast if API keys or the knowledge-base folder are missing, and creates the OpenAI-embeddings and Supabase clients.

Key choices:
- **1,536 embedding dimensions** — must match the `vector(1536)` column in Supabase, and stays under pgvector's 2,000-dim HNSW index limit.
- **2,000-char chunks with 200-char overlap** — roughly 500 tokens per chunk, with ~10% overlap so ideas aren't cut mid-thought.

**Step 2 (done outside the notebook, in the Supabase SQL editor):** the `documents` table — `id text` primary key (= `chunk_id`, which makes upserts idempotent), `content`, `metadata jsonb`, `embedding vector(1536)` — plus an HNSW cosine index and a `match_documents(query_embedding, match_count, filter)` function that powers every similarity search below. RLS is on; the notebook uses the service-role key.


In [31]:
# --- Standard library ---
import os                       # environment variables
import json                     # parse embedding strings returned by Supabase
from pathlib import Path        # walk the knowledge base folder
from itertools import groupby
# --- Config & validation ---
from dotenv import load_dotenv            # load API keys from .env
from pydantic import BaseModel, Field     # structured-output schemas

# --- Vector store & RAG pipeline ---
from supabase import create_client, Client                         # Supabase connection
from langchain_openai import OpenAIEmbeddings, ChatOpenAI          # embeddings + chat model
from langchain_core.prompts import ChatPromptTemplate              # prompt templates
from langchain_community.document_loaders import PyMuPDFLoader     # PDF document loader
from langchain_text_splitters import RecursiveCharacterTextSplitter # text splitter for chunking documents
from langchain_core.documents import Document
import re
import unicodedata
import tiktoken
# --- Utilities ---
from tqdm import tqdm           # progress bars
from collections import Counter # count occurrences of items in a list
from collections import defaultdict # dictionary with default values for missing keys
# --- Visualization (optional) ---
import numpy as np                    # embedding matrix
import pandas as pd # data manipulation and analysis
from sklearn.manifold import TSNE     # reduce 1536-D vectors to 2-D/3-D
import plotly.graph_objects as go     # interactive scatter plots


In [32]:
load_dotenv(override=True)

# --- LLM ---
MODEL = "gpt-6-luna"            # bare model name for ChatOpenAI

# --- Embeddings ---
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIMENSIONS = 1536            # must match vector(1536) in Supabase; under the 2,000-dim HNSW limit

# --- Supabase vector store ---
TABLE_NAME = "documents"               # pgvector table
MATCH_FUNCTION = "match_documents"     # SQL similarity-search function

# --- Knowledge base & chunking ---
KNOWLEDGE_BASE_PATH = Path("data-science-knowledge-base")
CHUNK_SIZE_CHARS = 2000                # ~500 tokens (≈4 chars/token in English)
CHUNK_OVERLAP_CHARS = 200              # ~10% overlap so ideas aren't cut mid-thought

# --- Sanity checks: fail fast with clear messages ---
for var in ("OPENAI_API_KEY", "SUPABASE_URL", "SUPABASE_KEY"):
    assert os.getenv(var), f"Missing {var} in .env"
assert KNOWLEDGE_BASE_PATH.exists(), f"Knowledge base not found: {KNOWLEDGE_BASE_PATH.resolve()}"

# --- Clients ---
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMENSIONS)
supabase: Client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

## Load & Clean the PDFs (Step 3)

**What this does:** turns the six reference PDFs into one clean LangChain `Document` per page, each with metadata used later for filtering and citations (`library`, `version`, `section`, `page`, `source_url`).

`clean_page()` handles what PDF extraction gets wrong:
- Unicode normalization (NFKC) plus a small translation map for Private-Use-Area glyphs (unmapped font characters that render as boxes).
- Parses the running page header (e.g. *"pandas 3.1.0 reference RESHAPING AND PIVOT TABLES"*) into `version` + `section` metadata, then strips it from the text. Chapter-opener pages have no header, so the title line is used instead.
- Captures `Source: https://...` lines as `source_url` for citations.

The second cell **canonicalizes section names**: running headers are UPPERCASE and openers are Title Case, and line-break hyphenation creates variants like `prac- tices` vs `Practices`. Matching on a lowercase alphanumeric key merges 72 variant spellings — without this, `groupby` would split sections apart in the chunking step. It also drops table-of-contents continuation pages.

**Result:** 1,040 pages loaded (TOC pages skipped), every page has a `section`, 74 pages carry a `source_url`.

**Lesson learned:** most RAG quality problems come from ingestion, not the model — always look at the extracted text before embedding.


In [33]:
HEADER_RE = re.compile(r"^(?P<library>[\w\-]+)\s+(?P<version>\d+\.\d+(?:\.\d+)?)\s+reference\s*(?P<section>.*)$")
SOURCE_RE = re.compile(r"Source:\s*(https?://\S+)")
PUA_MAP = str.maketrans({"\uee47": ": ", "\uee4e": "(", "\uee4f": ")","\uee55": "-", "\uee6c": "x"})
VERSION_RE = re.compile(r"\d+\.\d+(?:\.\d+)?")

def clean_page(doc):
    """ Normalize text, lift section /version into metadata, capture Source URLs."""
    text = unicodedata.normalize("NFKC", doc.page_content).translate(PUA_MAP)
    text = re.sub(r" {2,}", " ", text)
    lines = text.splitlines()
    nonblank = [k for k, l in enumerate(lines) if l.strip()]
    section = None

    if nonblank:
        i = nonblank[0]
        m = HEADER_RE.match(lines[i].strip())
        if m:
            doc.metadata["version"] = m["version"]
            section = m["section"].strip()
            lines.pop(i)
            if not section:
                j = next((k for k in range(i, len(lines)) if lines[k].strip()), None)
                if j is not None and lines[j].strip().isupper():
                    section = lines.pop(j).strip()
        else:
            src = next((k for k in nonblank[:4] if lines[k].startswith("Source:")), None)
            if src is not None:
                section = " ".join(lines[k].strip() for k in range(i, src) if lines[k].strip())
            elif len(lines[i].strip()) < 60:
                section = lines[i].strip()
    if section and section.isupper():
        section = section.title()
    doc.metadata["section"] = section or None

    if "version" not in doc.metadata:
        v = VERSION_RE.search(doc.metadata.get("title", ""))
        doc.metadata["version"] = v.group() if v else None
    
    text = "\n".join(lines)
    urls = SOURCE_RE.findall(text)
    if urls:
        doc.metadata["source_url"] = urls[0]
    doc.page_content = text.strip()
    return doc

docs, skipped = [], 0
for pdf in sorted(KNOWLEDGE_BASE_PATH.glob("*.pdf")):
    for d in PyMuPDFLoader(str(pdf)).load():
        if d.page_content.lstrip().startswith("Contents"): # table-of-contents pages
            skipped += 1
            continue
        d.metadata["library"] = pdf.stem.replace("_reference", "")
        d.metadata["page"] = d.metadata.get("page", 0) + 1
        docs.append(clean_page(d))

print(f"{len(docs)} pages loaded, {skipped} contents pages skipped")




1040 pages loaded, 6 contents pages skipped


In [34]:
# --- Canonicalize section names (opener wording wins; ignore case, spaces, hyphens) ---
def section_key(s):          # ignore case, spaces, hyphens and punctuation when matching
    return re.sub(r"[^a-z0-9]", "", s.casefold())

variants = defaultdict(set)
for d in docs:
    variants[(d.metadata["library"], section_key(d.metadata["section"]))].add(d.metadata["section"])
canon = {k: re.sub(r"(\w)- (\w)", r"\1\2",                       # rejoin "prac- tices" -> "practices"
                   next((v for v in vs if v != v.title()), next(iter(vs))))
         for k, vs in variants.items()}
for d in docs:
    d.metadata["section"] = canon[(d.metadata["library"], section_key(d.metadata["section"]))]
print(f"{sum(len(vs) > 1 for vs in variants.values())} sections had variant spellings")

before = Counter(d.metadata["library"] for d in docs)
docs = [d for d in docs if d.metadata["section"].casefold() != "contents"]
print("TOC continuation pages dropped:", dict(before - Counter(d.metadata["library"] for d in docs)))

72 sections had variant spellings
TOC continuation pages dropped: {'matplotlib': 2, 'numpy': 1, 'pandas': 2, 'plotly': 2, 'scikit-learn': 2, 'scipy': 1}


### Ingestion quality checks

Two diagnostic cells kept as regression checks for future re-runs:
1. Per-library page/character counts, the glued-word count (~104 remaining, all real class names like `HistGradientBoostingClassifier`), section/`source_url` coverage, and one sample page to eyeball.
2. A Private-Use-Area character scan (expect **0** after the glyph map) and a list of pages with no section (expect **none**).


In [35]:
print(pd.DataFrame([{"library": d.metadata["library"], 
                    "chars": len(d.page_content)} for d in docs]).groupby("library").agg(pages=("chars", "size"), 
                    chars=("chars", "sum")), "\n")
glued = Counter(w for d in docs for w in re.findall(r"\b[A-Za-z]{25,}\b", d.page_content))
print(f"Glued words: {sum(glued.values()):,} (target ≈ 100 )")
print(glued.most_common(10), "\n")

print(f"Pages with section: {sum(bool(d.metadata.get('section')) for d in docs)}/{len(docs)}")
print(f"Pages with source_url: {sum(bool(d.metadata.get('source_url')) for d in docs)} (target ≈ 74)")
print(Counter(d.metadata.get("section") for d in docs).most_common(15))
print(docs[50].metadata)
print(docs[50].page_content[:600])


              pages   chars
library                    
matplotlib      127  258863
numpy           149  286938
pandas          217  429038
plotly          155  300855
scikit-learn    231  565793
scipy           151  332100 

Glued words: 104 (target ≈ 100 )
[('HistGradientBoostingClassifier', 33), ('GradientBoostingRegressor', 21), ('GradientBoostingClassifier', 14), ('HistGradientBoostingRegressor', 10), ('TransformedTargetRegressor', 9), ('InconsistentVersionWarning', 4), ('LinearDiscriminantAnalysis', 3), ('FillBetweenPolyCollection', 2), ('ParquetCompressionOptions', 1), ('DatetimeScalarOrArrayConvertible', 1)] 

Pages with section: 1030/1030
Pages with source_url: 74 (target ≈ 74)
[('API quick reference', 295), ('Metrics and scoring', 44), ('Time series / date functionality', 37), ('Optimization (scipy.optimize)', 31), ('NumPy: the absolute basics', 30), ('Indexing and selecting data', 26), ('Linear models', 26), ('Ensembles: gradient boosting, random forests, bagging, voting, st

In [36]:
# 1) How wide spread are Private Use Area characters, and what surrounds them?
pua = Counter(ch for d in docs for ch in d.page_content if 0xE000 <= ord(ch)<= 0xF8FF)
print(f"PUA chars in body text: {sum(pua.values()):,}")
for ch, n in pua.most_common(20):
    ex = next(d.page_content for d in docs if ch in d.page_content)
    k = ex.index(ch)
    print(f"U+{ord(ch):04X} x{n:>5}: {ex[max(0, k-35):k+35]!r}")

# 2) Which pages have no section?
none_pages = [d for d in docs if not d.metadata.get("section")]
print("\nNo-section pages by library:", Counter(d.metadata["library"] for d in none_pages))
for d in none_pages[:6]:
    print(d.metadata["library"], "p.", d.metadata["page"], repr(d.page_content[:120]))

PUA chars in body text: 0

No-section pages by library: Counter()


## Chunking (Step 4)

**Why chunk:** embeddings represent a passage with a single vector, so each chunk should contain one coherent idea at a size the retriever can match precisely (~500 tokens here).

Two strategies are built and compared:
1. **Page-level** (`page_chunks`): split each page independently. Problem: the average page is ~2,150 chars, so most pages produce one full chunk *plus* a small leftover that cuts thoughts off at page breaks — a bimodal size distribution.
2. **Section-level** (`chunks`): first merge consecutive pages of the same `(library, section)` into "section runs" with `groupby`, *then* split — so chunks flow across page breaks. Citations become page ranges (`page`–`page_end`).

`chunk_report()` prints per-library stats, the size-distribution histogram, and a token/cost estimate; `summarize()` puts the two strategies side by side.

**Result:** 1,652 page-level vs **1,264 section-level** chunks; section-level has ~88% full-size chunks and much lower size variance. Embedding cost is ~$0.08 either way. Both sets are kept — page-level is the baseline for the Step 9 evaluation.

Every chunk gets a unique `chunk_id` (`library-pNNNN` / `library-sNNNN`) used as the database primary key, so re-running the pipeline *updates* rows instead of duplicating them. The sanity-check cell asserts the IDs don't collide across strategies.


In [37]:
enc = tiktoken.get_encoding("cl100k_base")
PRICE_PER_1M = 0.13 # Text-embedding-3_large, USD; confirm current OpenAI pricing.

def chunk_report(chunks, label, show=True):
    """ Per-Library stats, length distribution, token/cost estimate for one chunk set.
    Returns the per-chunk DataFrame so runs can be compared later. """

    df=pd.DataFrame({
        "library": [c.metadata["library"] for c in chunks],
        "chars": [len(c.page_content) for c in chunks],
        "tokens": [len(enc.encode(c.page_content)) for c in chunks],
    }).assign(strategy=label)

    if show:
        print(f"=== {label}. {len(df):,} chunks ===")
        print(df.groupby("library").agg(chunks=("chars", "size"),
                                        mean_chars=("chars", "mean")))
        print(df[["chars", "tokens"]].describe().round(0), "\n")
        print(f"Tiny chunks (<200 chars): {(df.chars < 200).sum()}")
        print(f"Total tokens: {df.tokens.sum():,} -> est.cost ${df.tokens.sum() /1e6 * PRICE_PER_1M:.3f}")
        go.Figure(go.Histogram(x=df.chars,
                                xbins=dict(start=0, end=CHUNK_SIZE_CHARS, size=50))).update_layout(title=f"Chunk Length Distribution - {label}",
                                                                                                   xaxis_title="Characters",
                                                                                                   yaxis_title="Count").show()
    return df

                

In [38]:
# --- Trim metadata to what retrieval and citations need ---


KEEP = ("library",
        "version",
        "section",
        "page",
        "source",
        "source_url")
for d in docs:
    d.metadata = {k: d.metadata.get(k) for k in KEEP}

# --- Split pages into overlapping chunks ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=["\n\n", "\n", " ", ""], # paragraph -> line -> word -> char
)
chunks = splitter.split_documents(docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = f"{c.metadata['library']}-p{i:04d}"

print(f"{len(docs)} pages-> {len(chunks)} chunks")
page_chunks = chunks

1030 pages-> 1631 chunks


In [39]:
page_df = chunk_report(page_chunks, "page-level")

=== page-level. 1,631 chunks ===
              chunks   mean_chars
library                          
matplotlib       200  1356.315000
numpy            218  1369.316514
pandas           316  1409.664557
plotly           218  1427.463303
scikit-learn     430  1393.341860
scipy            249  1400.726908
        chars  tokens
count  1631.0  1631.0
mean   1394.0   364.0
std     628.0   172.0
min      34.0    15.0
25%     790.0   197.0
50%    1685.0   425.0
75%    1956.0   493.0
max    2000.0  1101.0 

Tiny chunks (<200 chars): 14
Total tokens: 593,488 -> est.cost $0.077


In [40]:
# Join consecutive chunks with the same library + section, so chunks flow across page breaks
sections = []
for _, group in groupby(docs, key=lambda d: (d.metadata["library"], d.metadata["section"])):
    group = list(group)
    meta = dict(group[0].metadata) # page = first page of the run
    meta["page_end"] = group[-1].metadata["page"]
    sections.append(Document(page_content="\n".join(g.page_content for g in group), metadata=meta))

print(f"{len(docs)} pages -> {len(sections)} section runs")
chunks = splitter.split_documents(sections)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = f"{c.metadata['library']}-s{i:04d}"
print(f"{len(sections)} sections runs -> {len(chunks)} chunks")

1030 pages -> 92 section runs
92 sections runs -> 1243 chunks


In [41]:
ids = [c.metadata["chunk_id"] for c in page_chunks + chunks]
assert len(ids) == len(set(ids)), "duplicate chunk_ids"
print(page_chunks[0].metadata["chunk_id"], chunks[0].metadata["chunk_id"]) # expect matplotlib -p0000  matplotlib-s0000

matplotlib-p0000 matplotlib-s0000


In [42]:
section_df = chunk_report(chunks, "section-level")

=== section-level. 1,243 chunks ===
              chunks   mean_chars
library                          
matplotlib       151  1863.284768
numpy            164  1905.835366
pandas           243  1923.843621
plotly           174  1878.712644
scikit-learn     321  1921.928349
scipy            190  1903.284211
        chars  tokens
count  1243.0  1243.0
mean   1904.0   499.0
std     254.0   102.0
min     225.0    51.0
25%    1935.0   451.0
50%    1966.0   493.0
75%    1984.0   550.0
max    2000.0  1087.0 

Tiny chunks (<200 chars): 0
Total tokens: 619,918 -> est.cost $0.081


In [43]:
def summarize(df):
    return pd.Series({
        "chunks": len(df),
        "% < 1,600 chars": round((df.chars < 1600).mean() * 100, 1),
        "% 1,800-2000": round(df.chars.between(1800, 2000).mean() * 100, 1),
        "P25 chars": df.chars.quantile(.25),
        "median chars": df.chars.median(),
        "std chars": round(df.chars.std(), 1),
        "tiny (<200)": (df.chars < 200).sum(),
        "tokens": df.tokens.sum(),
        "est. cost $(USD)": round(df.tokens.sum() / 1e6 * PRICE_PER_1M, 3)
    })
display(pd.DataFrame({"page-level": summarize(page_df),
                      "section-level": summarize(section_df)}))
both = pd.concat([page_df, section_df])
fig = go.Figure()
for label, g in both.groupby("strategy"):
    fig.add_histogram(x=g.chars, 
                      name=label, 
                      opacity=0.6,
                      xbins=dict(start=0,
                                 end=CHUNK_SIZE_CHARS,
                                 size=50))
fig.add_vline(x=CHUNK_SIZE_CHARS,
              line_dash="dash",
              annotation_text="chunk_size")
fig.update_layout(barmode="overlay",
                  title="Effect of merging pages into section runs",
                  xaxis_title="Characters Per Chunk",
                  yaxis_title="Count"
                  ).show()

,page-level,section-level
chunks,1631.000,1243.000
"% < 1,600 chars",47.000,5.500
"% 1,800-2000",45.900,93.700
P25 chars,790.500,1935.000
median chars,1685.000,1966.000
std chars,628.300,254.200
tiny (<200),14.000,0.000
tokens,593488.000,619918.000
est. cost $(USD),0.077,0.081


## Embed & Store in Supabase (Step 5)

**What this does:** converts every chunk into a 1,536-dim vector and writes it to the `documents` table.

- **Smoke test first:** embed a single chunk and check the vector length (1536) and norm (~1.0 — OpenAI embeddings are unit-normalized, so cosine similarity is just a dot product).
- `upsert_chunks()` works in batches of 100: embed the batch, then upsert rows keyed on `chunk_id`. Each row's metadata is tagged with its `strategy` (`"section"` or `"page"`) so both chunking strategies share one table and are selected with a metadata filter.
- **Delete-by-strategy before loading:** an upsert never deletes, so stale rows from earlier runs (e.g. after re-chunking changed the IDs) are cleared first.
- The verification cell counts rows per library **on the server** (PostgREST truncates `res.data` at 1,000 rows, but `count="exact"` is exact) and compares against `section_df`.

**Result:** 1,264 section-level rows, matching the chunk counts exactly.


In [44]:
# --- Embedding client + single-chunk smoke test ---
# (Clients are re-created here so this section can be re-run on its own after a kernel restart.)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, 
                             dimensions=EMBEDDING_DIMENSIONS)
supabase: Client = create_client(os.environ["SUPABASE_URL"],
                                 os.environ["SUPABASE_KEY"]) # service-role key

v = embeddings.embed_query(chunks[0].page_content)
print(len(v), round(sum(x * x for x in v) ** 0.5, 3)) # expect 1536 and ~1.0 (unit-normalized)

1536 1.0


In [45]:
# --- Upsert section-level chunks in batches---
BATCH = 100

def upsert_chunks(chunk_list, strategy):
    for start in tqdm(range(0, len(chunk_list), BATCH),desc=strategy):
        batch = chunk_list[start:start + BATCH]
        vectors = embeddings.embed_documents([c.page_content for c in batch])
        rows = [{
            "id": c.metadata["chunk_id"],
            "content": c.page_content,
            "metadata": {**c.metadata, "strategy":strategy},
            "embedding": vec} for c, vec in zip(batch, vectors)]
        supabase.table(TABLE_NAME).upsert(rows).execute()
supabase.table(TABLE_NAME).delete().eq("metadata->>strategy", "section").execute() # clear old section rows
upsert_chunks(chunks,"section")

section: 100%|██████████| 13/13 [00:23<00:00,  1.85s/it]


In [46]:
# --- Verify upserted chunks ---

res = supabase.table(TABLE_NAME).select("library", count="exact").eq("metadata->>strategy", "section").execute()
print(f"Rows in Supabase: {res.count} (expect {len(chunks)})")

for lib in sorted(section_df.library.unique()):
    n = supabase.table(TABLE_NAME).select("id", count="exact", head=True).eq("library", lib).execute().count
    print(f"{lib:<14}{n:>5} (expect {(section_df.library == lib).sum()})")


Rows in Supabase: 1243 (expect 1243)
matplotlib      351 (expect 151)
numpy           382 (expect 164)
pandas          559 (expect 243)
plotly          392 (expect 174)
scikit-learn    751 (expect 321)
scipy           439 (expect 190)


## Retrieval Sanity Test (Step 6)

**What this does:** before wiring up the LLM, check that similarity search alone returns sensible documentation. `retrieve()` embeds a question, calls the `match_documents` RPC with a metadata filter, and shows the top-k hits with scores.

Five probe questions (one per library, plus one deliberately ambiguous) check that:
- the top-1 hit comes from the expected library and a sensible section,
- similarity falls gradually from rank 1 to 5 (a cliff would suggest a lucky hit),
- a metadata filter (`library="plotly"`) still returns all k rows.

**Result:** clear questions hit the right library 5/5 with top-1 similarity ~0.56–0.68; the ambiguous secondary-y-axis question stays flat around 0.45. That gap is what motivates the **0.50 confidence cutoff** used in the RAG chain (Step 8) and tuned in the evaluation (Step 9).


In [47]:
# --- Retrieval Sanity Test ---
def retrieve(question, k=5, **filters):
    qv = embeddings.embed_query(question)
    res = supabase.rpc(MATCH_FUNCTION, {"query_embedding": qv,
                                         "match_count": k,
                                         "filter": {"strategy": "section", **filters}}).execute()
    return pd.DataFrame([{"sim": round(r["similarity"], 3),
                          "id": r["id"],
                          "library": r["metadata"]["library"],
                          "section": r["metadata"]["section"],
                          "pages": f'{r["metadata"]["page"]}-{r["metadata"].get("page_end")}',
                          "preview": r["content"][:80].replace("\n", " ")
                          } for r in res.data]) 
probes = {
    "How do I Pivot a DataFrame from long to wide format?": "pandas",
    "How do I tune hyperparameters with cross-validation?": "scikit-learn",
    "How do I minimize a function with constraints?": "scipy",
    "How do I broadcast arrays of different shapes?": "numpy",
    "How do I add a secondary y-axis to a plot?": None # ambiguous: matplotlib or plotly
    }
for q, expected in probes.items():
    print(f"\nQ: {q} (expect: {expected})")
    display(retrieve(q))

print("\nFiltered: same question, plotly only")
display(retrieve("How do i add a secondary y-axis to a plot?", library="plotly"))


Q: How do I Pivot a DataFrame from long to wide format? (expect: pandas)


,sim,id,library,section,pages,preview
0,0.587,pandas-s0410,pandas,Reshaping and pivot tables,100-111,Reshaping and pivot tables Source: https://pan...
1,0.525,pandas-s0414,pandas,Reshaping and pivot tables,100-111,") df = pd.DataFrame(np.random.randn(8, 4), ind..."
2,0.508,pandas-s0547,pandas,API quick reference,149-220,"• If margins is True, will also normalize marg..."
3,0.483,pandas-s0546,pandas,API quick reference,149-220,"... ""small"", ... ""large"", ... ""small"", ... ""sm..."
4,0.482,pandas-s0544,pandas,API quick reference,149-220,"Examples >>> df = pd.DataFrame( ... { ... ""key..."



Q: How do I tune hyperparameters with cross-validation? (expect: scikit-learn)


,sim,id,library,section,pages,preview
0,0.622,scikit-learn-s0806,scikit-learn,Cross-validation,58-72,When evaluating different settings (”hyperpara...
1,0.608,scikit-learn-s0805,scikit-learn,Cross-validation,58-72,Cross-validation Source: https://scikit-learn....
2,0.606,scikit-learn-s0825,scikit-learn,Tuning hyper-parameters,73-82,Tuning hyper-parameters Source: https://scikit...
3,0.526,scikit-learn-s0808,scikit-learn,Cross-validation,58-72,>>> n_samples = X.shape[0] >>> cv = ShuffleSpl...
4,0.521,scikit-learn-s0738,scikit-learn,Getting started,6-9,"... >>> X, y = make_regression(n_samples=1000,..."



Q: How do I minimize a function with constraints? (expect: scipy)


,sim,id,library,section,pages,preview
0,0.561,scipy-s1057,scipy,Optimization (scipy.optimize),5-35,"... options={'xatol': 1e-8, 'disp': True}) Opt..."
1,0.549,scipy-s1065,scipy,Optimization (scipy.optimize),5-35,"For medium-size problems, for which the storag..."
2,0.546,scipy-s1056,scipy,Optimization (scipy.optimize),5-35,Optimization (scipy.optimize) Source: https://...
3,0.540,scipy-s1066,scipy,Optimization (scipy.optimize),5-35,This optimization problem has the unique solut...
4,0.529,scipy-s1081,scipy,Optimization (scipy.optimize),5-35,3: |F(x)| = 27.3682; step 1; tol 0.0348109 4: ...


... (2 outputs left out)


Q: How do I add a secondary y-axis to a plot? (expect: None)


,sim,id,library,section,pages,preview
0,0.464,plotly-s0612,plotly,Subplots,44-54,fig.show() Subplots with Shared X-Axes The sha...
1,0.460,matplotlib-s0011,matplotlib,Quick start guide,6-15,One caveat about categorical plotting is that ...
2,0.451,plotly-s0569,plotly,The figure data structure,8-13,which case default values will be inferred. Mu...
3,0.451,plotly-s0611,plotly,Subplots,44-54,"row=1, col=2) fig.show() Customizing Subplot A..."
4,0.447,plotly-s0616,plotly,Subplots,44-54,"fig.add_trace(go.Scatter3d(x=[2, 3, 1], y=[0, ..."



Filtered: same question, plotly only


,sim,id,library,section,pages,preview
0,0.476,plotly-s0612,plotly,Subplots,44-54,fig.show() Subplots with Shared X-Axes The sha...
1,0.471,plotly-s0569,plotly,The figure data structure,8-13,which case default values will be inferred. Mu...
2,0.466,plotly-s0611,plotly,Subplots,44-54,"row=1, col=2) fig.show() Customizing Subplot A..."
3,0.450,plotly-s0616,plotly,Subplots,44-54,"fig.add_trace(go.Scatter3d(x=[2, 3, 1], y=[0, ..."
4,0.447,plotly-s0649,plotly,Line charts,81-89,"y1_lower = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] y1_l..."


## Visualize the Embedding Space (Step 7)

**What this does:** t-SNE squashes the 1,536-dim vectors down to 2-D/3-D while preserving *local neighborhoods* — a visual sanity check that chunks from the same library land near each other, i.e. the embeddings actually capture topic.

The fetch cell pages through Supabase 1,000 rows at a time (PostgREST's response cap) and `json.loads` each vector, because pgvector returns embeddings as strings.

**Result:** ~97.5% of chunks have same-library nearest neighbors in the layout. The overlaps that do exist are semantically real, not projection noise (they appear in both 2-D and 3-D): **matplotlib ↔ plotly** (both plotting) and **numpy ↔ scipy** (shared array math).

**Caveat:** t-SNE preserves neighborhoods, not global distances — don't read meaning into the gaps *between* clusters.


In [48]:
# --- Fetch embeddings from Supabase (paged past the 1,000-row cap) ---
rows, start = [], 0
while True:
    page = (supabase.table(TABLE_NAME)
            .select("id, library, section:metadata->>section, embedding")
            .eq("metadata->>strategy", "section")
            .range(start, start + 999).execute().data)
    rows += page
    if len(page) < 1000:
        break
    start += 1000

X = np.array([json.loads(r["embedding"]) for r in rows])
meta = pd.DataFrame(rows).drop(columns="embedding")
print(X.shape) # expect (1264, 1536)

(1243, 1536)


In [49]:
# --- t-SNE: 1536-D -> 2D/3D, colored by library ---
def plot_tsne(X, meta, dims=2, perplexity=30, seed=42):
    """ Fit t-SNE (cosine) to `dims` dimensions and plot, colored by library.
     Returns the coordinates so they can be reused without refitting. """
    coords =TSNE(n_components=dims, 
                 metric="cosine",
                 perplexity=perplexity,
                 init="pca",
                 random_state=seed).fit_transform(X)
    axes = ["x", "y", "z"][:dims]
    df = meta.assign(**dict(zip(axes, coords.T)))

    fig = go.Figure()
    for lib, g in df.groupby("library"):
        common = dict(mode="markers", 
                      name=lib,
                      text=g.section + "<br>" + g.id,
                      hovertemplate="%{text}<extra>" + lib + "</extra>")

        if dims == 3:
            fig.add_scatter3d(x=g.x,
                              y=g.y,
                              z=g.z,
                              marker=dict(size=3, opacity=0.7),
                              **common)
        else:
            fig.add_scatter(x=g.x,
                            y=g.y,
                            marker=dict(size=5, opacity=0.7),
                            **common)
    fig.update_layout(title=f"Section-level chunk embeddings (t-SNE {dims}D, cosine, perplexity={perplexity})",
                   width=900,
                   height=700)
    fig.show()
    return df

In [50]:
tsne_2d = plot_tsne(X, meta, dims=2)

In [51]:
tsne_3d = plot_tsne(X, meta, dims=3)

## RAG Chain — Answers with Citations (Step 8)

**What this does:** glues retrieval and the LLM together into `ask(question)`:
1. **Retrieve** the top-5 section chunks for the question.
2. **Confidence gate:** if the best similarity is below `MIN_SIM` (0.40), skip the LLM and answer *"couldn't find this"* — refusing beats hallucinating (the near-miss sources are still returned for transparency).
3. **Answer:** number the chunks into context blocks, and ask the model to answer *only* from them. Pydantic structured output (`Answer`) forces the model to return both the answer and the block numbers it used — which makes the citations trustworthy rather than decorative.
4. **Cite:** the cited blocks are mapped back to deduplicated sources (library, version, section, page range, URL, similarity).

The smoke test asks one pandas question, one ambiguous plotting question, and one question that *isn't in the knowledge base* ("capital of France") to confirm the refusal path works.


In [ ]:
# --- RAG Chain: retrieve -> answer with citations---

MIN_SIM = 0.40 # skip the LLM if the best match is weaker than this (tuned with the step 9 cutoff sweep)
TOP_K = 5
REFUSAL = "I couldn't find this in the documentation with enough confidence."

class Answer(BaseModel):
    # in_context comes FIRST so the model judges coverage before it starts writing an answer.
    in_context: bool = Field(description= """True only if the context blocks directly answer the question.
        False if the question is about a library, tool or feature the blocks don't cover, 
        even when the blocks discuss something similar.""")
    answer: str =Field(description="""Answer using only the context. Include a short code example when helpful.
    In if_context is False, say in one sentence what the context covers instead.""")
    cited: list[int] = Field(description="""Numbers of the context blocks the answer relies on, e.g. [1, 3]. " 
    Empty if in_context is False.""")

# USD per 1M tokens: verify at developers.openai.com/api/docs/pricing
PRICES = {'gpt-6-luna': {'input': 0.10, 'cached': 0.01, 'output': 0.50}}
NO_USAGE = {'input_tokens': 0, 'output_tokens': 0, 'cost': 0.0}

# include_raw=True returns {'raw': AIMessage, 'parsed': Answer, 'parsing_error':...}
llm = ChatOpenAI(model=MODEL).with_structured_output(Answer, include_raw=True)

def llm_cost(usage, model=MODEL):
    p = PRICES[model]
    cached = (usage.get('input_token_details') or {}).get('cache_read', 0) or 0
    return ((usage['input_tokens'] -cached) * p['input'] + cached * p['cached'] + usage['output_tokens'] * p['output']) / 1e6

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a documentation assistant for data science libraries. Answer ONLY from the numbered
    context blocks. Each block is labeled with the library it comes from. If the question asks about a library
    or tool that none of the blocks come from, set if_context to False. Do not answer it with a different
    library's equivalent. Never make up an answer. List the block numbers you used only in `cited`; do not
    include block numbers in the answer text."""),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def cite(hit):
    m = hit["metadata"]
    pages = str(m["page"]) if m["page"] == m.get("page_end") else f'{m["page"]}-{m.get("page_end")}'
    return {"library": m["library"], "version": m["version"], "section": m["section"],
            "pages": pages, "url": m.get("source_url"), "sim": round(hit["similarity"], 3)}

def dedupe(sources):
    out = {}
    for s in sources:
        out.setdefault((s["library"], s["section"]),s)
    return list(out.values())

def search(question, k=TOP_K, qv=None, **filters):
    qv = qv if qv is not None else embeddings.embed_query(question) # reuse cached vectors when given
    return supabase.rpc(MATCH_FUNCTION, {'query_embedding': qv,
                                         'match_count': k,
                                         'filter': {'strategy': 'section', **filters}
                                         }).execute().data

def ask(question, k=TOP_K, qv=None,**filters):
    hits = search(question, k, qv=qv, **filters)
    near = dedupe([cite(h) for h in hits[:3]])
    if not hits or hits[0]['similarity'] < MIN_SIM: # layer 1: similarity gate
        # gate refusal
        return {'answer': REFUSAL,
                'sources': near,
                'confident': False,
                'refused_by': 'gate',
                'context': None,
                **NO_USAGE}

    context = "\n\n".join(f"[{i}] {h['metadata']['library']} · {h['metadata']['section']}\n{h['content']}"
                        for i, h in enumerate(hits, 1))
    
    res = (prompt | llm).invoke({'context': context, 'question': question})
    if res['parsed'] is None:
        raise ValueError(res['parsing_error'])

    out, usage = res['parsed'], res['raw'].usage_metadata
    u = {'input_tokens': usage['input_tokens'], 
         'output_tokens': usage['output_tokens'],
         'cost':llm_cost(usage)}

    if not out.in_context: # layer 2: LLM says "not covered"
        # LLM refusal
        return {'answer': REFUSAL,
                'sources': near,
                'confident': False,
                'refused_by': 'llm',
                'llm_note': out.answer,
                'context': context,
                **u}
    
    used = [hits[i - 1] for i in out.cited if 1 <= i <= len(hits)]
    # answered
    return {'answer': out.answer,
            'sources': dedupe([cite(h) for h in used]),
            'confident': True,
            'refused_by': None,
            'context': context,
            **u}

    
    
    

In [63]:
# --- RAG smoke test: one clear hit per library boundary + one unanswerable ---

for q in ["how do I pivot a DataFrame from long to wide format?",
          "how do I add a secondary y-axis to a plot?",
          "What is the capital of France?"
          ]:
    r = ask(q)
    print(f"\nQ: {q}\nconfident={r['confident']}\n\n{r['answer']}\n")
    display(pd.DataFrame(r["sources"]))


Q: how do I pivot a DataFrame from long to wide format?
confident=True

Use `DataFrame.pivot`, specifying the column to use as the new index, the column whose unique values become columns, and the values to fill those columns. For example: `wide = df.pivot(index="date", columns="variable", values="value")`. If your data contains duplicate index/column combinations, use `pivot_table` to aggregate them instead.



,library,version,section,pages,url,sim
0,pandas,3.1.0,Reshaping and pivot tables,100-111,https://pandas.pydata.org/docs/user_guide/resh...,0.579



Q: how do I add a secondary y-axis to a plot?
confident=True

In Plotly, overlay a second y-axis on the existing chart and assign a trace to it. The context notes that axes can be overlaid to create dual-axis charts, but does not provide the exact configuration syntax. It also describes using `make_subplots` and `update_yaxes` to customize subplot axes.



,library,version,section,pages,url,sim
0,plotly,7.1.0,The figure data structure,8-13,https://plotly.com/python/figure-structure/,0.456
1,plotly,7.1.0,Subplots,44-54,https://plotly.com/python/subplots/,0.449



Q: What is the capital of France?
confident=False

I couldn't find this in the documentation with enough confidence.



,library,version,section,pages,url,sim
0,plotly,7.1.0,Scatter plots,72-80,https://plotly.com/python/line-and-scatter/,0.105
1,numpy,2.5.4,NumPy: the absolute basics,7-36,https://numpy.org/doc/stable/user/absolute_beg...,0.096


## Evaluation — Hit Rate & MRR (Step 9)

**What this does:** measures retrieval quality objectively, and settles the page-vs-section chunking question with numbers instead of anecdotes.

The eval set has **45 answerable questions** labeled with their expected `(library, section)` — including deliberate boundary cases (e.g. a NaN-filling question that could be pandas *or* scikit-learn) — plus **5 unanswerable** ones. Questions are embedded **once** and reused, because OpenAI embeddings aren't perfectly deterministic and re-embedding adds noise between runs.

**The metrics, in plain terms:**
- **hit@k** — fraction of questions where the correct section appears in the top *k* results. hit@1 = "right answer first", hit@5 = "right answer on the first page".
- **MRR** (mean reciprocal rank) — averages 1/rank of the correct section: 1.0 for rank 1, 0.5 for rank 2, 0 for a miss. Rewards putting the right chunk *high*, not just somewhere in the top k.
- **library@1** — did the top hit at least come from the right library?

Page-level chunks are loaded as `strategy="page"` so both strategies are scored on identical questions, then the misses are listed for inspection.

The final cell is the **confidence-cutoff sweep**: plotting answerable vs unanswerable top-1 similarities and, for each candidate threshold, how many good questions survive vs how many bad ones get rejected — this is how `MIN_SIM` is chosen.


In [54]:
# --- Eval set: question -> expected (library, section) ---
EVAL = [
    # pandas
    ("How do I fill missing values with the previous value?", "pandas", "Working with missing data"),
    ("How do I combine two DataFrames on a common key column?", "pandas", "Merge, join, concatenate and compare"),
    ("How do I compute the mean of a column for each group?", "pandas", "Group by: split-apply-combine"),
    ("How do I resample daily data to monthly frequency?", "pandas", "Time series / date functionality"),
    ("Why does chained assignment no longer modify my DataFrame?", "pandas", "Copy-on-Write and migration guide"),
    ("How do I select rows that meet a condition with .loc?", "pandas", "Indexing and selecting data"),
    ("How do I unpivot a wide DataFrame into long format?", "pandas", "Reshaping and pivot tables"),
    # numpy
    ("What is the difference between a copy and a view of an array?", "numpy", "Copies and views"),
    ("How do I create an array of evenly spaced numbers?", "numpy", "Array creation"),
    ("How does indexing an array with an array of integers work?", "numpy", "Indexing on ndarrays"),
    ("How do I store values as 32-bit floats?", "numpy", "Data types"),
    ("How can I avoid Python loops to make array code faster?", "numpy", "Writing performant NumPy code"),
    ("When are two array shapes compatible for arithmetic?", "numpy", "Broadcasting"),
    ("How do I apply an element-wise function and use its reduce method?", "numpy", "Universal functions (ufunc) basics"),
    # scipy
    ("How do I fit a normal distribution to data and get its parameters?", "scipy", "Probability distributions"),
    ("How do I numerically integrate a function between two limits?", "scipy", "Integration (scipy.integrate)"),
    ("How do I draw a smooth cubic spline through data points?", "scipy", "1-D interpolation"),
    ("How do I solve a linear system Ax = b?", "scipy", "Linear algebra (scipy.linalg)"),          # numpy boundary
    ("How do I find nearest neighbors quickly with a KD-tree?", "scipy", "Spatial data structures and algorithms (scipy.spatial)"),
    ("Which sparse format is best for fast row slicing?", "scipy", "Sparse arrays (scipy.sparse)"),
    ("How do I estimate a probability density with Gaussian kernels?", "scipy", "Kernel density estimation"),
    ("How do I find the minimum of a function with constraints?", "scipy", "Optimization (scipy.optimize)"),
    # scikit-learn
    ("How do I scale features to zero mean and unit variance?", "scikit-learn", "Preprocessing data"),
    ("How do I chain a scaler and a model so preprocessing happens inside cross-validation?", "scikit-learn", "Pipelines and composite estimators"),
    ("How do I fill NaN values before training a model?", "scikit-learn", "Imputation of missing values"),   # pandas boundary
    ("How do I compute ROC AUC for a classifier?", "scikit-learn", "Metrics and scoring"),
    ("How do I save a trained model to disk?", "scikit-learn", "Model persistence"),
    ("How does a random forest differ from gradient boosting?", "scikit-learn", "Ensembles: gradient boosting, random forests, bagging, voting, stacking"),
    ("How do I search a grid of hyperparameter values?", "scikit-learn", "Tuning hyper-parameters"),
    ("What is data leakage and how do I avoid it?", "scikit-learn", "Common pitfalls and recommended practices"),
    # matplotlib
    ("How do I place a legend outside the axes?", "matplotlib", "Legend guide"),
    ("How do I use a logarithmic y-axis?", "matplotlib", "Axis scales"),
    ("How do I stop subplot titles and labels from overlapping?", "matplotlib", "Constrained layout guide"),
    ("How do I draw an arrow pointing at a data point?", "matplotlib", "Annotations"),
    ("Which colormap should I use for sequential data?", "matplotlib", "Choosing colormaps"),
    ("What is the difference between pyplot and the object-oriented interface?", "matplotlib", "Matplotlib application interfaces (pyplot vs. object-oriented)"),
    # plotly
    ("How do I export a figure as a PNG file?", "plotly", "Static image export"),             # matplotlib boundary
    ("How do I make a candlestick chart of stock prices?", "plotly", "Candlestick charts"),
    ("How do I create a grid of subplots with make_subplots?", "plotly", "Subplots"),
    ("How do I apply a dark theme to all my figures?", "plotly", "Theming and templates"),
    ("How do I save an interactive chart as an HTML file?", "plotly", "Interactive HTML export"),
    ("How do I make a box plot with Plotly Express?", "plotly", "Box plots"),
    ("How do I change how figures display in a Jupyter notebook?", "plotly", "Displaying figures (renderers)"),
    # unanswerable: not in the knowledge base
    ("How do I train a neural network in PyTorch?", None, None),
    ("What is the capital of France?", None, None),
    ("How do I connect to PostgreSQL with SQLAlchemy?", None, None),
    ("How do I deploy a Flask web app?", None, None),
    ("How do I tokenize text with spaCy?", None, None),
    # casual phrasing, sections not yet covered ---
    ("my x axis dates are all squished together, how do i control where the tick marks go", "matplotlib", "Axis ticks"),
    ("can I set default font size once so every plot uses it", "matplotlib", "Customizing with style sheets and rcParams"),
    ("plotting a million points is super slow, any way to speed it up?", "matplotlib", "Performance"),
    ("hex code vs named color, what formats does matplotlib accept for colors", "matplotlib", "Specifying colors"),
    ("put some math/latex text on my chart", "matplotlib", "Text in Matplotlib"),
    ("what makes a numpy array different from a python list", "numpy", "What is NumPy?"),
    ("quick way to see a scatter plot where dot size shows a third variable", "plotly", "Scatter plots"),
    ("color a grid of numbers by value like a correlation matrix", "plotly", "Heatmaps"),
    ("change a trace's color after the figure is already built", "plotly", "Creating and updating figures"),
    ("throw out useless columns before fitting, keep only the predictive ones", "scikit-learn", "Feature selection"),
    ("is my model overfitting? plot train vs test score as I add more data", "scikit-learn", "Validation and learning curves"),
    ("ridge vs lasso, which penalty zeroes out coefficients", "scikit-learn", "Linear models"),
    ("test whether my sample mean is significantly different from 5", "scipy", "Analysing one sample"),
    # Out of scope questions
    ("How do I add a callback to a Plotly Dash dropdown?", None, None),      # plotly-adjacent
    ("How do I make a pairplot with seaborn?", None, None),                  # matplotlib-adjacent
    ("How do I use early stopping in XGBoost?", None, None),                 # sklearn-adjacent
    ("How do I get p-values from an OLS regression in statsmodels?", None, None),  # scipy.stats-adjacent
    ("How do I read a CSV lazily with Polars?", None, None),                 # pandas-adjacent
    ("How do I parallelize a groupby with Dask?", None, None),               # pandas-adjacent
    ("How do I move a tensor to the GPU in PyTorch?", None, None),           # numpy-adjacent
    ("How do I forecast a time series with Prophet?", None, None),           # pandas time series-adjacent
    ("How do I write a SQL window function for a running total?", None, None),
    ("How do I undo my last git commit?", None, None),
]
# PROPOSED
# Extra acceptable (library, section) pairs for questions with more than one correct answer
ALSO = {
    "How do I apply a dark theme to all my figures?": [("matplotlib", "Customizing with style sheets and rcParams")],
    "How do I export a figure as a PNG file?": [("matplotlib", "Introduction to Figures")],
    "How do I solve a linear system Ax = b?": [("numpy", "API quick reference")],
    "How do I fill NaN values before training a model?": [("pandas", "Working with missing data")],
    "How do I stop subplot titles and labels from overlapping?": [("plotly", "Subplots")],
}
eval_df = pd.DataFrame(EVAL, columns=["question", "library", "section"])
eval_df["targets"] = [[(r.library, r.section)] + ALSO.get(r.question, []) if pd.notna(r.library) else []
                      for r in eval_df.itertuples()]

# Every label must exist in the chunks (catches typos and canonicalization differences)
valid = {(c.metadata["library"], c.metadata["section"]) for c in chunks}
bad = {t for ts in eval_df.targets for t in ts} - valid
assert not bad, f"Unknown labels: {bad}"
assert not (set(ALSO) - set(eval_df.question)), "ALSO key doesn't match a question"

eval_df["qv"] = embeddings.embed_documents(eval_df.question.tolist())     # embed ONCE, reuse for every run
print(f"{len(eval_df)} questions ({eval_df.library.notna().sum()} answerable, "
      f"{(eval_df.targets.str.len() > 1).sum()} multi-label)")


71 questions (56 answerable, 5 multi-label)


In [55]:
# --- Load the page-level baseline so both strategies can be compared on the same questions ---
supabase.table(TABLE_NAME).delete().eq("metadata->>strategy", "page").execute() # clear old page rows first
upsert_chunks(page_chunks, "page") # ~1,631 chunks, ~$0.08

page: 100%|██████████| 17/17 [00:30<00:00,  1.79s/it]


In [56]:
# --- Hit rate@k and MRR: page vs section ---
def evaluate(strategy, k=10): # Evaluate retrieval strategy with hit rate@k and MRR 
    rows = []
    for r in eval_df.itertuples():
        hits = supabase.rpc(MATCH_FUNCTION, {'query_embedding': r.qv,
                                             'match_count': k,
                                             'filter': {'strategy':strategy}}).execute().data
        rank = next((i for i, h in enumerate(hits, 1) if (h['metadata']['library'],
                                                          h['metadata']['section']) in r.targets), np.nan)
        rows.append({'question': r.question,
                     'library': r.library,
                     'section': r.section,
                     'answerable': r.library is not None,
                     'rank': rank,
                     'top_sim': hits[0]['similarity'],
                     'top_lib': hits[0]['metadata']['library'],
                     'top_section': hits[0]['metadata']['section'],
                     'target_libs': {lib for lib, _ in r.targets}})
    return pd.DataFrame(rows).assign(strategy=strategy)

def metrics(res, ks=(1, 3, 5, 10)):
    a = res[res.answerable]
    m = {f'hit@{k}': (a['rank'] <= k).mean() for k in ks} # NaN (miss) <= k is False
    m['MRR'] = (1 / a['rank']).fillna(0).mean()
    m['library@1'] = pd.Series([t in libs for t, libs in zip(a.top_lib, a.target_libs)]).mean()
    return pd.Series(m).round(3)

results = pd.concat([evaluate('page'), evaluate('section')], ignore_index=True)
display(results.groupby('strategy')[results.columns.drop('strategy')].apply(metrics).T)
display(results.query("answerable and strategy == 'section' and (rank > 3 or rank != rank)")
        [['question', 'library', 'rank', 'top_lib', 'top_section', 'top_sim']]) # misses to inspect



strategy,page,section
hit@1,0.839,0.804
hit@3,0.946,0.964
hit@5,1.000,1.000
hit@10,1.000,1.000
MRR,0.897,0.880
library@1,0.964,0.964


,question,library,rank,top_lib,top_section,top_sim
89,How do I find nearest neighbors quickly with a...,scipy,5.0,scipy,API quick reference,0.462990
103,How do I stop subplot titles and labels from o...,matplotlib,4.0,matplotlib,Text in Matplotlib,0.592085


In [57]:
# --- Confidence-cutoff sweep: where should MIN_SIM sit? ---
sec = results[results.strategy == "section"]

fig = go.Figure()
for ans, g in sec.groupby("answerable"):
    fig.add_histogram(x=g.top_sim,
                      name="answerable" if ans else "unanswerable",
                      opacity=0.6,
                      xbins=dict(start=0,
                                 end=1,
                                 size=0.025))
# show once, after BOTH histograms are added (was inside the loop: figure rendered twice,
# the first time with only one trace)
fig.update_layout(barmode="overlay",
                  title="Top-1 Similarity: Answerable vs Unanswerable",
                  xaxis_title="Top-1 Cosine Similarity").show()

# For each candidate cutoff: fraction of answerable questions kept vs unanswerable rejected
display(pd.DataFrame([{"cutoff": t,
                       "answerable kept": (sec[sec.answerable].top_sim >= t).mean(),
                       "unanswerable rejected": (sec[~sec.answerable].top_sim < t).mean()}
                      for t in np.arange(0.20, 0.61, 0.05).round(2)]))


,cutoff,answerable kept,unanswerable rejected
0,0.20,1.000000,0.133333
1,0.25,1.000000,0.200000
2,0.30,1.000000,0.333333
3,0.35,1.000000,0.466667
4,0.40,1.000000,0.733333
5,0.45,0.946429,0.800000
6,0.50,0.803571,0.866667
7,0.55,0.517857,1.000000
8,0.60,0.214286,1.000000


In [58]:
# --- Which out-of-scope questions get pas MIN_SIM, and what do they retrieve? ---
display(results.query("not answerable and strategy =='section'").sort_values('top_sim', ascending=False)[['question', 'top_sim', 'top_lib', 'top_section']].assign(passes_gate=lambda d: d.top_sim >= MIN_SIM))


,question,top_sim,top_lib,top_section,passes_gate
134,How do I use early stopping in XGBoost?,0.530157,scikit-learn,"Ensembles: gradient boosting, random forests, ...",True
137,How do I parallelize a groupby with Dask?,0.514332,pandas,Group by: split-apply-combine,True
133,How do I make a pairplot with seaborn?,0.470730,plotly,Plotly Express,True
135,How do I get p-values from an OLS regression i...,0.421242,scipy,API quick reference,True
139,How do I forecast a time series with Prophet?,0.388294,pandas,10 minutes to pandas,False
116,How do I connect to PostgreSQL with SQLAlchemy?,0.379517,pandas,API quick reference,False
132,How do I add a callback to a Plotly Dash dropd...,0.367361,plotly,Creating and updating figures,False
136,How do I read a CSV lazily with Polars?,0.367123,pandas,API quick reference,False
140,How do I write a SQL window function for a run...,0.340564,pandas,Group by: split-apply-combine,False
117,How do I deploy a Flask web app?,0.301133,plotly,Getting started with Plotly,False


In [59]:
# --- Paired page vs section: does the same question rank better under one strategy? ---
from scipy.stats import wilcoxon
wide = (results[results.answerable].pivot(index='question', columns='strategy', values='rank').fillna(11)) # miss =worse than k=10
wide['diff'] = wide['page'] - wide['section'] # > 0 -> section ranked it better
print(wide['diff'].apply(lambda d: 'section better' if d > 0 else 'page better' if d < 0 else 'tie').value_counts().to_string())
display(wide[wide['diff'] != 0].sort_values('diff'))
nz = wide.loc[wide['diff'] != 0, 'diff']
if len(nz) >= 6:
    print(f"Wilcoxon signed-rank: p = {wilcoxon(nz).pvalue:.3f} (n = {len(nz)} non-tied questions)")

diff
tie               44
page better        6
section better     6


strategy,page,section,diff
question,,,
How do I stop subplot titles and labels from overlapping?,1.0,4.0,-3.0
How do I find nearest neighbors quickly with a KD-tree?,3.0,5.0,-2.0
can I set default font size once so every plot uses it,1.0,3.0,-2.0
How can I avoid Python loops to make array code faster?,2.0,3.0,-1.0
How do I create an array of evenly spaced numbers?,1.0,2.0,-1.0
How do I draw an arrow pointing at a data point?,1.0,2.0,-1.0
How do I chain a scaler and a model so preprocessing happens inside cross-validation?,4.0,3.0,1.0
color a grid of numbers by value like a correlation matrix,4.0,3.0,1.0
"my x axis dates are all squished together, how do i control where the tick marks go",2.0,1.0,1.0


Wilcoxon signed-rank: p = 0.690 (n = 12 non-tied questions)


In [61]:
# --- Two-layer refusal: similarity gate + LLM in_context, full eval set ---
rows = []
for r in eval_df.itertuples():
    out = ask(r.question, qv=r.qv) # cached vectors, so no re-embedding noise
    rows.append({
        'question': r.question,
        'answerable': r.library is not None,
        'refused_by': out['refused_by'],
        'answer': out['answer'],
        'llm_note': out.get('llm_note'),
        'sources': out['sources'],
        **{k: out[k] for k in ('input_tokens', 'output_tokens', 'cost')}})

qa = pd.DataFrame(rows).assign(refused=lambda d: d.refused_by.notna())

display(qa.groupby('answerable').refused_by.value_counts(dropna=False)) 
display(qa.query("answerable and refused")[['question', 'refused_by', 'llm_note']]) # false refusals (cost)
display(qa.query("not answerable and not refused")[['question', 'answer']]) # remaining leaks

# --- Cist if this eval run ---
called =qa[qa.input_tokens > 0]
print(f"LLM calls: {len(called)} of {len(qa)} (gate skipped {len(qa) - len(called)})")
print(f"Tokens in/out: {called.input_tokens.sum():,} / {called.output_tokens.sum():,}")
print(f"Total: ${qa.cost.sum():.4f} | per LLM call: ${called.cost.mean():.6f}"
      f"| per 1,000 questions: ${called.cost.mean() * 1000:.2f}")
display(qa.groupby('refused_by', dropna=False)[['input_tokens', 'output_tokens', 'cost']].agg(['mean', 'sum']).round(6))

answerable  refused_by
False       gate          11
            llm            4
True        NaN           56
Name: count, dtype: int64

,question,refused_by,llm_note


,question,answer


LLM calls: 60 of 71 (gate skipped 11)
Tokens in/out: 172,541 / 10,522
Total: $0.0225 | per LLM call: $0.000375| per 1,000 questions: $0.38


input_tokens         output_tokens             cost          
                   mean     sum          mean    sum      mean       sum
refused_by                                                              
gate               0.00       0      0.000000      0  0.000000  0.000000
llm             2899.25   11597     72.750000    291  0.000326  0.001305
NaN             2874.00  160944    182.696429  10231  0.000379  0.021210

In [67]:
# --- Step 9d-1: answer every answerable question with BOTH chunking strategies ---
ans_eval = eval_df[eval_df.library.notna()]
gen = []
for r in tqdm(ans_eval.itertuples(), total=len(ans_eval)):
    for strat in ("page", "section"):
        out = ask(r.question, qv=r.qv, strategy=strat)          # **filters overrides the 'section' default
        gen.append({"question": r.question, "strategy": strat,
                    **{k: out.get(k) for k in ("answer", "context", "refused_by",
                                               "input_tokens", "output_tokens", "cost")}})
gen = pd.DataFrame(gen)
display(gen.groupby("strategy")[["input_tokens", "output_tokens", "cost"]].sum())
display(gen.groupby("strategy").refused_by.value_counts(dropna=False))   # any refusals = automatic weak answers

100%|██████████| 56/56 [05:34<00:00,  5.97s/it]


,input_tokens,output_tokens,cost
strategy,,,
page,131062,9763,0.006207
section,161560,9507,0.006384


strategy  refused_by
page      NaN           56
section   NaN           56
Name: count, dtype: int64

In [68]:
# --- Step 9d-2: blind pairwise judge ---
from typing import Literal

JUDGE_MODEL = "gpt-6-sol"   # stronger than the answering model, to avoid self-preference bias
PRICES[JUDGE_MODEL] = {"input": 2.00, "cached": 2.00, "output": 10.00}  # cached price unverified, so set to input price (conservative)

class Judgment(BaseModel):
    # reason FIRST: the judge reasons before it commits to scores (same idea as in_context first)
    reason: str = Field(description="One or two sentences comparing the answers.")
    faithful_a: bool = Field(description="True if every claim in Answer A is supported by Context A.")
    faithful_b: bool = Field(description="True if every claim in Answer B is supported by Context B.")
    complete_a: int = Field(ge=1, le=5, description="How fully Answer A answers the question. 1 = not at all, 5 = fully.")
    complete_b: int = Field(ge=1, le=5, description="How fully Answer B answers the question. 1 = not at all, 5 = fully.")
    preferred: Literal["A", "B", "tie"] = Field(description="Which answer better serves the user overall.")

judge = ChatOpenAI(model=JUDGE_MODEL).with_structured_output(Judgment, include_raw=True)

judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """You grade answers from a documentation assistant. Each answer was written ONLY from its own context.
Judge faithfulness of each answer against ITS OWN context. Judge completeness against the question.
Do not prefer an answer because it is longer or appears first. Prefer 'tie' when the answers are equally good."""),
    ("human", "Question: {question}\n\n--- Context A ---\n{ctx_a}\n\n--- Answer A ---\n{ans_a}\n\n"
              "--- Context B ---\n{ctx_b}\n\n--- Answer B ---\n{ans_b}"),
])

def judge_run(run, seed=42):
    rng = np.random.default_rng(seed + run)                     # different but reproducible A/B order per run
    rows = []
    for q, g in tqdm(gen.groupby("question", sort=False), desc=f"run {run}"):
        g = g.set_index("strategy")
        swap = rng.random() < 0.5                               # blind + randomized position
        sa, sb = ("section", "page") if swap else ("page", "section")
        a, b = g.loc[sa], g.loc[sb]
        res = (judge_prompt | judge).invoke({"question": q,
                "ctx_a": a.context or "(no context: refused)", "ans_a": a.answer,
                "ctx_b": b.context or "(no context: refused)", "ans_b": b.answer})
        if res["parsed"] is None:
            raise ValueError(res["parsing_error"])
        j = res["parsed"]
        rows.append({"run": run, "question": q, "swap": swap, "preferred": j.preferred,
                     "winner": {"A": sa, "B": sb, "tie": "tie"}[j.preferred],
                     f"faithful_{sa}": j.faithful_a, f"faithful_{sb}": j.faithful_b,
                     f"complete_{sa}": j.complete_a, f"complete_{sb}": j.complete_b,
                     "reason": j.reason, "judge_cost": llm_cost(res["raw"].usage_metadata, JUDGE_MODEL)})
    return pd.DataFrame(rows)

In [69]:
# --- Step 9d-3: judge, then score ---
from scipy.stats import binomtest

judged = pd.concat([judged] + [judge_run(r) for r in (1, 2)], ignore_index=True)  # start with 1 run; change to range(3) once it looks right

# 1. Who wins? (majority across runs per question; split votes -> tie)
maj = judged.groupby("question").winner.agg(lambda s: s.mode()[0] if s.mode().size == 1 else "tie")
counts = maj.value_counts(); print(counts.to_string())
n = counts.get("page", 0) + counts.get("section", 0)
if n:
    print(f"Sign test, ties dropped: p = {binomtest(counts.get('section', 0), n).pvalue:.3f} (n = {n})")

# 2. Quality
display(judged[["faithful_page", "faithful_section", "complete_page", "complete_section"]].mean().round(3))

# 3. Judge sanity checks
print("Position bias, share preferring A (≈ share preferring B means no bias):",
      judged.preferred.value_counts(normalize=True).round(3).to_dict())
if judged.run.nunique() > 1:
    print("Same winner in every run:", (judged.groupby("question").winner.nunique() == 1).mean().round(3))

# 4. Cost / quality per dollar
c = gen.groupby("strategy").cost.sum()
print(f"Answer cost per strategy: {c.round(4).to_dict()} | judge cost: ${judged.judge_cost.sum():.3f}")
display(judged.query("winner != 'tie'")[["question", "winner", "reason"]])   # read these: the judge's reasons

run 2: 100%|██████████| 56/56 [02:15<00:00,  2.43s/it]

winner
tie        33
page       14
section     9
Sign test, ties dropped: p = 0.405 (n = 23)


faithful_page       0.970
faithful_section    0.952
complete_page       4.756
complete_section    4.661
dtype: float64

Position bias, share preferring A (≈ share preferring B means no bias): {'tie': 0.548, 'B': 0.25, 'A': 0.202}
Same winner in every run: 0.625
Answer cost per strategy: {'page': 0.0062, 'section': 0.0064} | judge cost: $2.015


,question,winner,reason
1,How do I combine two DataFrames on a common ke...,page,Both answers give the correct merge syntax and...
8,How do I create an array of evenly spaced numb...,section,Both answers faithfully explain how to use lin...
9,How does indexing an array with an array of in...,page,Both answers correctly explain how multiple in...
16,How do I draw a smooth cubic spline through da...,section,Both answers faithfully explain and plot a cub...
18,How do I find nearest neighbors quickly with a...,page,Answer A gives a working construction-and-quer...
...,...,...,...
160,what makes a numpy array different from a pyth...,page,Both answers accurately explain the core diffe...
161,quick way to see a scatter plot where dot size...,section,Both answers faithfully show how to map a thir...
162,color a grid of numbers by value like a correl...,page,Both answers show how to color a numeric grid ...
164,"throw out useless columns before fitting, keep...",page,Both answers are faithful to their contexts an...


In [70]:
# --- Step 9d-4: inspect the judge ===
pd.set_option("display.max_colwidth", None)
# Unfaithful answers: the most important failures, more than wins
display(judged.loc[~judged.faithful_page | ~judged.faithful_section, 
                   ['run', 'question', 'faithful_page', 'faithful_section', 'reason']])
# Position bias: does the winner follow the strategy or the slot?
display(pd.crosstab(judged.swap, judged.preferred, margins=True))

,run,question,faithful_page,faithful_section,reason
1,0,How do I combine two DataFrames on a common key column?,True,False,Both answers give the correct merge syntax and explain the join options. Context B explicitly supports the claim that the default is an inner join; Context A shows the merge example and lists inner joins but does not establish that default.
9,0,How does indexing an array with an array of integers work?,False,False,"Both answers correctly explain how multiple integer index arrays select paired positions, but neither explains the single-index-array case the question most directly asks about. Both also state that the result has the broadcast index shape without noting that unindexed dimensions remain; B is slightly more useful because it clarifies paired selection versus selecting every combination."
21,0,How do I find the minimum of a function with constraints?,False,True,"Both answers explain how to use `minimize` for constrained optimization. B gives a more concrete example and shows how to retrieve the result, but its claim that `jac` is optional for SLSQP constraints is not supported by its context, which specifies dictionaries with `type`, `fun`, and `jac`. A is fully supported by its context."
54,0,"ridge vs lasso, which penalty zeroes out coefficients",False,True,"Both identify Lasso as the method that can set coefficients exactly to zero. Answer A additionally says ridge uses an L2 penalty, but Context A does not establish that; Answer B stays within its context."
65,1,How does indexing an array with an array of integers work?,False,True,"A explains both a single integer index array and multiple index arrays, including the shape of the result. B gives a useful pairwise example, but its unqualified claim that the result has the indices’ broadcast shape does not hold when dimensions remain unindexed."
84,1,How do I search a grid of hyperparameter values?,True,False,"Both answers explain how to fit GridSearchCV over a parameter grid. Answer A names `best_params_`, which is not documented in its provided context; Answer B stays within its context and includes the supported held-out evaluation advice."
100,1,can I set default font size once so every plot uses it,True,False,"Answer A is supported by its context, but it sets sizes for specific plot elements rather than one default font size. Answer B addresses the question directly, but its context does not establish that `font.size` is a valid rcParam, so its central instruction is not supported by its own context."
106,1,color a grid of numbers by value like a correlation matrix,True,False,Both answers show how to color a numeric grid as a heatmap. Answer B stays within its context; Answer A adds a specific colorscale and parameter not supported by its context.
133,2,How do I find the minimum of a function with constraints?,False,True,"Both answers show how to pass constraints and bounds to `minimize`, but neither defines all the variables in its code example. Answer A says the `jac` key is optional for SLSQP constraint dictionaries, whereas its context describes dictionaries with `type`, `fun`, and `jac` keys; Answer B stays within its context."
140,2,How do I search a grid of hyperparameter values?,True,False,"Both answers explain how to run an exhaustive grid search. Answer A also gives the context-supported advice to evaluate on held-out data, while Answer B names `best_params_`, an attribute not documented in its provided context."


preferred,A,B,tie,All
swap,,,,
False,20,15,47,82
True,14,27,45,86
All,34,42,92,168


In [71]:
   gen.to_parquet("judge_answers.parquet"); judged.to_parquet("judge_results.parquet")

## Next — Step 10: Streamlit App

Remaining work: wrap `ask()` in a chat UI with streamed answers, source citations (section + page range + URL), and a library filter dropdown. Also worth revisiting:
- Pin `MIN_SIM` using the cutoff sweep above.
- Known knowledge-base gap: secondary y-axis / `twinx` isn't covered well by the PDFs.
